# Azure Service Bus Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/azure_service_bus/service_bus_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/azure_service_bus/service_bus_demo.ipynb)

## Business Scenario

Service Bus queues and topics carry operational events across distributed systems. You need contract-driven validation before storing them.

## Value Proposition

- Reliable queue ingestion
- Schema validation and quarantine
- Consistent event processing at scale

---

## Goals

1. Connect to Service Bus
2. Validate messages
3. Persist clean records


## 🚀 Step 1: Setup Azure Service Bus

Before running this notebook, you need:
1. An Azure Service Bus namespace
2. A queue or topic/subscription
3. Azure credentials configured (via `az login` or Managed Identity)

Set your environment variable:
```bash
export AZURE_SERVICEBUS_NAMESPACE="my-namespace.servicebus.windows.net"
```

## 📝 Step 2: Review the Contract

Our contract defines the expected Service Bus message schema and quality rules.

In [ ]:
with open('service_bus_contract.yaml', 'r') as f:
    print("📄 Service Bus Contract:")
    print("-----------------------")
    print(f.read())

## ▶️ Step 3: Start the Service Bus Receiver

This will connect to your Service Bus queue and start processing messages.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(
    contract="service_bus_contract.yaml",
    framework="bytewax"
)

# Start in background
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 Service Bus receiver started!")
print("Waiting for notification events...")
time.sleep(5)

## 🧪 Step 4: Send Test Messages

Let's send some test notification events to the queue.

In [ ]:
from azure.servicebus import ServiceBusClient, ServiceBusMessage
from azure.identity import DefaultAzureCredential
import json
import os
from datetime import datetime

namespace = os.getenv("AZURE_SERVICEBUS_NAMESPACE")
queue_name = "notification-queue"

if namespace:
    credential = DefaultAzureCredential()
    servicebus_client = ServiceBusClient(
        fully_qualified_namespace=namespace,
        credential=credential
    )
    
    sender = servicebus_client.get_queue_sender(queue_name=queue_name)
    
    test_notifications = [
        {
            "notificationId": "NOTIF-001",
            "userId": "USER-123",
            "type": "email",
            "message": "Your order has been shipped!",
            "priority": "high",
            "timestamp": datetime.now().isoformat(),
            "metadata": {"orderId": "ORD-456"}
        },
        {
            "notificationId": "NOTIF-002",
            "userId": "USER-789",
            "type": "push",
            "message": "New message from support",
            "priority": "medium",
            "timestamp": datetime.now().isoformat()
        }
    ]
    
    with sender:
        messages = [ServiceBusMessage(json.dumps(notif)) for notif in test_notifications]
        sender.send_messages(messages)
        print(f"📤 Sent {len(messages)} notifications to Service Bus")
    
    servicebus_client.close()
    print("\n✅ Test messages sent! Check LakeLogic logs...")
else:
    print("ℹ️  Set AZURE_SERVICEBUS_NAMESPACE to send test messages")

## 📊 Step 5: Verify Results

Check the materialized Delta table.

In [ ]:
import polars as pl
import time

# Wait for processing
time.sleep(3)

try:
    df = pl.read_delta("./data/bronze/azure_notifications/")
    print("📂 Processed Notifications:")
    print(df)
    
    print("\n📊 Priority Distribution:")
    print(df.group_by("priority").count())
except Exception as e:
    print(f"ℹ️  No data yet: {e}")

## 🎉 Summary

You just:
- ✅ Connected to Azure Service Bus
- ✅ Validated notification events against a contract
- ✅ Materialized events to Delta Lake

This pattern enables **reliable asynchronous messaging** for Azure-native applications!